In [1]:
# =====================
# TensorFlow / GPU
# =====================
import os
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print("GPU available:", gpus)
else:
    print("No GPU, using CPU")

if os.getenv("CUDA_VISIBLE_DEVICES") is None:
    gpu_num = 0  # 使用 CPU 可设为 ""
    os.environ["CUDA_VISIBLE_DEVICES"] = f"{gpu_num}"

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"


# =====================
# Scientific stack
# =====================
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.transform import Rotation as R

%matplotlib inline


# =====================
# Mitsuba / DrJit
# =====================
import mitsuba as mi
import drjit as dr


# =====================
# Sionna RT
# =====================
from sionna.rt import (
    load_scene,
    PlanarArray,
    Transmitter,
    Receiver,
    Camera,
    PathSolver,
    ITURadioMaterial,
    SceneObject,
    AntennaPattern,
    register_antenna_pattern,
)


# =====================
# Custom utils / scenes
# =====================
import sionnautils
from sionnautils.custom_scene import list_scenes, get_scene


# =====================
# Project-specific
# =====================
from Engine_V3 import Engine


# =====================
# Misc
# =====================
import json
from pathlib import Path
import yaml


2026-02-06 21:32:13.432661: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]



Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:
from sionnautils.custom_scene import list_scenes, get_scene
scenes = list_scenes()
print(scenes)

scene_path, map_data = get_scene('nyu_tandon')
for k, v in map_data.items():
    print(f'{k}: {v}')

scene = load_scene(scene_path,merge_shapes=True)

floor = scene.get('ground')
# print(f'Floor material: {floor.radio_material.name}')
floor.radio_material = ITURadioMaterial("itu_concrete",
                                "concrete",
                                thickness=0.01,
                                color=(0.5, 0.5, 0.5))

scene.remove("itu_wet_ground")

for name, obj in scene.objects.items():
    print(f'{name:<15}{obj.radio_material.name}')
# scene.render(camera=my_cam, num_samples=512)

scene.radio_materials

['nyu_tandon']
bbox_lat: [40.69012764197041, 40.699120858029595]
bbox_long: [-73.99156687083165, -73.97970552916836]
address: 5 MetroTech Center, Brooklyn, NY 11201
descr: NYU Tandon campus
2026-02-06 21:32:18 WARN  [HDRFilm] Monochrome mode enabled, setting film output pixel format to 'luminance' (was rgb).
no-name-1      itu_marble
ground         itu_concrete


{'itu_marble': ITURadioMaterial type=marble
                  eta_r=7.074
                  sigma=0.018
                  thickness=0.100
                  scattering_coefficient=0.000
                  xpd_coefficient=0.000,
 'itu_concrete': ITURadioMaterial type=concrete
                  eta_r=5.240
                  sigma=0.123
                  thickness=0.010
                  scattering_coefficient=0.000
                  xpd_coefficient=0.000}

In [3]:
import yaml
import numpy as np

def normalize_config(obj):
    """Recursively normalize:
    - numeric strings (incl. scientific notation) -> float
    - 'np.pi', 'np.e', 'np.inf' -> numpy constants
    """
    if isinstance(obj, dict):
        return {k: normalize_config(v) for k, v in obj.items()}

    if isinstance(obj, list):
        return [normalize_config(v) for v in obj]

    if isinstance(obj, str):
        s = obj.strip()

        # numpy constants
        if s == "np.pi":
            return np.pi
        if s == "np.e":
            return np.e
        if s in ("np.inf", "inf"):
            return np.inf
        if s in ("-np.inf", "-inf"):
            return -np.inf

        # numeric strings (supports "15.0e9", "200.0e6", "3.5E9", etc.)
        try:
            return float(s)
        except ValueError:
            return obj  # keep as string if not numeric

    return obj  # int/float/bool/None stay as-is


with open("config.yaml", "r") as f:
    cfg = yaml.safe_load(f)

cfg = normalize_config(cfg)


In [4]:
# ============================================================
# 4) Mitsuba inside-building test
# ============================================================
def is_inside_building_mitsuba(scene, point, direction=np.array([0.37, 0.23, 0.90]), max_hits=50):
    """
    Ray parity test: odd #hits -> inside.
    direction uses a non-axis-aligned vector to reduce degeneracy.

    NOTE: Most reliable if meshes are watertight. If your scene isn't watertight,
    consider a multi-direction voting version later.
    """
    point = np.asarray(point, dtype=np.float32)
    direction = np.asarray(direction, dtype=np.float32)
    direction = direction / (np.linalg.norm(direction) + 1e-12)

    ray = mi.Ray3f(o=mi.Point3f(point), d=mi.Vector3f(direction))
    scene_mi = scene._scene  # Sionna wraps Mitsuba internally

    count = 0
    for _ in range(max_hits):
        si = scene_mi.ray_intersect(ray, active=True)
        if not si.is_valid():
            break
        count += 1
        ray.o = si.p + 1e-4 * ray.d  # avoid self-hit

    return (count % 2 == 1)


def ue_inside_building(scene, cfg, position, rotation, check_center_first=True):
    """
    position: (3,) or (n,3)
    rotation: (3,) or (n,3)  radians, order [yaw, pitch, roll] for "zyx"
    cfg["ue"]["rx_loc_pos"]: (n_rx,3) UE-local RX offsets
    """
    rx_offset = np.asarray(cfg["ue"]["rx_loc_pos"], dtype=np.float32)

    position = np.atleast_2d(np.asarray(position, dtype=np.float32))
    rotation = np.atleast_2d(np.asarray(rotation, dtype=np.float32))

    n_ue = position.shape[0]
    for i in range(n_ue):
        if check_center_first and is_inside_building_mitsuba(scene, position[i]):
            return True

        r = R.from_euler("zyx", rotation[i], degrees=False)
        rx_positions = r.apply(rx_offset) + position[i]  # (n_rx,3)

        if any(is_inside_building_mitsuba(scene, p) for p in rx_positions):
            return True

    return False

In [5]:
i = 806

output_dir = Path(cfg["route"]["folder"])
file_path = output_dir / f"routes_{i:04d}.npz"

data = np.load(file_path)
print(data["positions"].shape)   # (n_ue, n_step, 3)
print(data["rotations"].shape)


(1, 1200, 3)
(1, 1200, 3)


In [6]:
data["rotations"]

array([[[ 0.8397281 , -0.04283386,  0.        ],
        [ 1.222779  ,  0.08974369,  0.        ],
        [ 1.4176078 ,  0.24842465,  0.        ],
        ...,
        [-2.3129036 ,  1.3925127 ,  0.        ],
        [-2.5876899 ,  1.3015836 ,  0.        ],
        [-2.9335587 ,  1.3420932 ,  0.        ]]], dtype=float32)

In [7]:
bad = 0
for ue in range(data["positions"].shape[0]):
    for t in range(data["positions"].shape[1]):
        if ue_inside_building(scene, cfg, data["positions"][ue, t], data["rotations"][ue, t]):
            bad += 1
print("bad steps:", bad)

bad steps: 0


In [8]:
data["rotations"]

array([[[ 0.8397281 , -0.04283386,  0.        ],
        [ 1.222779  ,  0.08974369,  0.        ],
        [ 1.4176078 ,  0.24842465,  0.        ],
        ...,
        [-2.3129036 ,  1.3925127 ,  0.        ],
        [-2.5876899 ,  1.3015836 ,  0.        ],
        [-2.9335587 ,  1.3420932 ,  0.        ]]], dtype=float32)

In [9]:
data["walk_rotations"]

array([[[ 2.847479  ,  0.        ,  0.        ],
        [ 2.849671  ,  0.        ,  0.        ],
        [ 2.8723342 ,  0.        ,  0.        ],
        ...,
        [-0.40274206,  0.        ,  0.        ],
        [-0.37781483,  0.        ,  0.        ],
        [-0.38018268,  0.        ,  0.        ]]], dtype=float32)

In [10]:
data["spin_rotations"]

array([[[-2.007751  , -0.04283386,  0.        ],
        [-1.6268919 ,  0.08974369,  0.        ],
        [-1.4547265 ,  0.24842465,  0.        ],
        ...,
        [-1.9101615 ,  1.3925127 ,  0.        ],
        [-2.209875  ,  1.3015836 ,  0.        ],
        [-2.553376  ,  1.3420932 ,  0.        ]]], dtype=float32)

In [11]:
scene.rx_array = PlanarArray(num_rows=1,
                              num_cols=1,
                              vertical_spacing=0.5,
                              horizontal_spacing=0.5,
                              pattern="iso",
                              polarization="V")


for idx_step in range(len(data["positions"][0])):
    # print(idx_step)
    rx = Receiver(name=f"rx-{idx_step}",
              position=np.array(data["positions"][0][idx_step]),
              display_radius=1)
    scene.remove(f"rx-{idx_step}")
    scene.add(rx)

scene.preview()

In [12]:
import socket, os
print("HOST:", socket.gethostname())
print("DISPLAY:", os.environ.get("DISPLAY"))


HOST: fc-hopping-gpu-6c8b8b7488-zlrxv
DISPLAY: :8
